In [ ]:
import pandas as pd
from pathlib import Path
from functools import reduce

base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
results_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/dphil_paper_2/results/connectivity_results")


In [ ]:
# helper to read and rename norm_scenario_pct with a suffix
def load_norm(path, suffix):
    return (
        pd.read_csv(path, usecols=["catchment_uid", "norm_scenario_pct"])
        .rename(columns={"norm_scenario_pct": f"norm_scenario_pct_{suffix}"})
        .drop_duplicates(subset=["catchment_uid"])
    )

# baseline
df_b05 = load_norm(
    base_path / "dphil_papers/dphil_paper_2/results/connectivity_results/catchment_connectivity__baseline__lambda_0.5.csv",
    "baseline_0.5",
)
df_b5 = load_norm(
    base_path / "dphil_papers/dphil_paper_2/results/connectivity_results/catchment_connectivity__baseline__lambda_5.csv",
    "baseline_5",
)

df_b50 = load_norm(
    base_path / "dphil_papers/dphil_paper_2/results/connectivity_results/catchment_connectivity__baseline__lambda_50.csv",
    "baseline_50",
)

# partial
df_p05 = load_norm(
    base_path / "dphil_papers/dphil_paper_2/results/connectivity_results/catchment_connectivity__partial__lambda_0.5.csv",
    "partial_0.5",
)
df_p5 = load_norm(
    base_path / "dphil_papers/dphil_paper_2/results/connectivity_results/catchment_connectivity__partial__lambda_5.csv",
    "partial_5",
)

df_p50 = load_norm(
    base_path / "dphil_papers/dphil_paper_2/results/connectivity_results/catchment_connectivity__partial__lambda_50.csv",
    "partial_50",
)

# long_timeframe (keep plantations)
df_lt_kp_05 = load_norm(
    base_path / "dphil_papers/dphil_paper_2/results/connectivity_results/catchment_connectivity__long_timeframe_keep_plantations__lambda_0.5.csv",
    "long_timeframe_keep_plantations_0.5",
)
df_lt_kp_5 = load_norm(
    base_path / "dphil_papers/dphil_paper_2/results/connectivity_results/catchment_connectivity__long_timeframe_keep_plantations__lambda_5.csv",
    "long_timeframe_keep_plantations_5",
)

df_lt_kp_50 = load_norm(
    base_path / "dphil_papers/dphil_paper_2/results/connectivity_results/catchment_connectivity__long_timeframe_keep_plantations__lambda_50.csv",
    "long_timeframe_keep_plantations_50",
)

# long_timeframe
df_lt05 = load_norm(
    base_path / "dphil_papers/dphil_paper_2/results/connectivity_results/catchment_connectivity__long_timeframe__lambda_0.5.csv",
    "long_timeframe_0.5",
)
df_lt5 = load_norm(
    base_path / "dphil_papers/dphil_paper_2/results/connectivity_results/catchment_connectivity__long_timeframe__lambda_5.csv",
    "long_timeframe_5",
)

df_lt50 = load_norm(
    base_path / "dphil_papers/dphil_paper_2/results/connectivity_results/catchment_connectivity__long_timeframe__lambda_50.csv",
    "long_timeframe_50",
)

# merge all
merged = reduce(
    lambda left, right: left.merge(right, on="catchment_uid", how="outer"),
    [df_b05, df_b5, df_b50, df_p05, df_p5, df_p50, df_lt_kp_05, df_lt_kp_5, df_lt_kp_50, df_lt05, df_lt5, df_lt50],
)

merged["norm_scenario_pct_baseline_avg"] = merged[
    ["norm_scenario_pct_baseline_0.5", "norm_scenario_pct_baseline_5", "norm_scenario_pct_baseline_50"]
].mean(axis=1)
merged["norm_scenario_pct_partial_avg"] = merged[
    ["norm_scenario_pct_partial_0.5", "norm_scenario_pct_partial_5", "norm_scenario_pct_partial_50"]
].mean(axis=1)
merged["norm_scenario_pct_long_timeframe_keep_plantations_avg"] = merged[
    ["norm_scenario_pct_long_timeframe_keep_plantations_0.5", "norm_scenario_pct_long_timeframe_keep_plantations_5", "norm_scenario_pct_long_timeframe_keep_plantations_50"]
].mean(axis=1)

merged["baseline_to_restoration_increase"] = (
    merged["norm_scenario_pct_partial_avg"] - merged["norm_scenario_pct_baseline_avg"]
)

merged["long_timeframe_minus_partial"] = (
    merged["norm_scenario_pct_long_timeframe_keep_plantations_avg"] - merged["norm_scenario_pct_partial_avg"]
)


merged["connectivity_increase_rank"] = merged["baseline_to_restoration_increase"].rank(
    method="dense", ascending=False
).astype(int)


merged.to_csv(
    results_path / "catchment_connectivity_norm_pct_all.csv",
    index=False,
)
merged.head()
